# Notebook 08 — FT + RAG (LoRA + FAISS)

Objectif: generer `results/ft_rag_predictions.json` avec un modele fine-tune et un contexte recupere via FAISS.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/llm-integration-study/'

!pip uninstall -y unsloth unsloth_zoo trl transformers peft accelerate bitsandbytes xformers
!pip install --no-cache-dir -U git+https://github.com/unslothai/unsloth-zoo.git
!pip install --no-cache-dir -U "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q sentence-transformers faiss-cpu tqdm


In [ ]:
import os, json, time
import numpy as np
import torch
import faiss
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from unsloth import FastLanguageModel

PROCESSED_PATH = os.path.join(BASE_PATH, 'data', 'processed')
RESULTS_PATH   = os.path.join(BASE_PATH, 'results')
FAISS_PATH     = os.path.join(BASE_PATH, 'models', 'faiss_index')
LORA_PATH      = os.path.join(BASE_PATH, 'models', 'lora_adapter_ft')
os.makedirs(RESULTS_PATH, exist_ok=True)

EMBED_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
TOP_K = 5
MAX_SEQ_LEN = 1024
MAX_CTX_CHARS = 1400

with open(os.path.join(PROCESSED_PATH, 'test.json'), 'r', encoding='utf-8') as f:
    test_data = json.load(f)
with open(os.path.join(FAISS_PATH, 'metadata.json'), 'r', encoding='utf-8') as f:
    corpus_meta = json.load(f)
index = faiss.read_index(os.path.join(FAISS_PATH, 'index.faiss'))
embed_model = SentenceTransformer(EMBED_MODEL)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

def retrieve_top_k(question, k=TOP_K):
    q_emb = embed_model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    chunks = [corpus_meta[i] for i in indices[0] if i < len(corpus_meta)]
    return chunks, scores[0].tolist()

def generate_ft_rag(question, chunks, max_new_tokens=320):
    context_block = '\n\n'.join([f"[Extrait {i+1}] {c.get('text','')[:900]}" for i, c in enumerate(chunks)])
    context_block = context_block[:MAX_CTX_CHARS]
    prompt = (
        '### Instruction: Reponds en francais de facon concise et factuelle en te basant sur le contexte fourni.\n'
        f'### Context: {context_block}\n'
        f'### Input: {question}\n'
        '### Response:'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    t0 = time.time()
    with torch.no_grad():
        gen = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=20,
            do_sample=False,
            repetition_penalty=1.16,
            no_repeat_ngram_size=4,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    lat = round((time.time() - t0) * 1000)
    ans = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    for stop in ['\n### Instruction:', '\n### Input:', '\n### Response:', '\n### Context:']:
        if stop in ans:
            ans = ans.split(stop)[0].strip()
    return ans, lat

ft_rag_predictions = []
for item in tqdm(test_data, desc='FT+RAG (LoRA + FAISS)'):
    q = item.get('question', '')
    g = item.get('answer', '')
    chunks, scores = retrieve_top_k(q, TOP_K)
    pred, lat = generate_ft_rag(q, chunks)
    ft_rag_predictions.append({
        'pair_id': item.get('pair_id', ''),
        'question': q,
        'predicted_answer': pred,
        'true_answer': g,
        'latency_ms': lat,
        'retrieved_chunks': [f"{c.get('doc_id','')}#{c.get('chunk_idx','')}" for c in chunks],
        'retrieval_scores': scores,
        'method': 'ft_rag',
        'dataset_type': item.get('dataset_type', ''),
        'question_type': item.get('question_type', ''),
    })

outp = os.path.join(RESULTS_PATH, 'ft_rag_predictions.json')
with open(outp, 'w', encoding='utf-8') as f:
    json.dump(ft_rag_predictions, f, ensure_ascii=False, indent=2)
lats = [p['latency_ms'] for p in ft_rag_predictions if p['latency_ms'] > 0]
print(f'Sauvegarde: {outp}')
print(f'N={len(ft_rag_predictions)} | latence moyenne={np.mean(lats):.0f} ms' if lats else f'N={len(ft_rag_predictions)}')
